# Lab 15 — Markov Chain 시뮬레이션 (선택)

**확률통계 · Week 15 · 부산대학교 정보컴퓨터공학부**

---

### 오늘의 목표

1. Random walk의 퍼짐이 $\sqrt{t}$ 로 자라는 것을 확인한다.
2. **행렬을 곱하는 것**과 **오래 돌려서 세는 것**이 같은 답을 준다는 것을 확인한다.
3. **PageRank**를 직접 계산해본다.

⏱ **예상 소요 시간: 30분** · 📌 **제출 의무 없음** (기말고사 주간)

> 이 랩은 확률통계 II로 넘어가는 다리다. 관심 있으면 해보고,
> 시험 준비가 급하면 나중에 봐도 좋다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260302)
print("준비 완료")

## Part 1. Random walk

매 걸음 ±1로 움직인다. 1,000명이 동시에 걸으면?

### 실습 1

In [ ]:
N_WALK, T = 1000, 500

steps = rng.choice([-1, 1], size=(N_WALK, T))
# TODO 1: 걸음을 누적해 위치를 만드세요.  힌트: np.cumsum(steps, axis=1)
paths = np.zeros((N_WALK, T))
t = np.arange(1, T + 1)

spread = paths.std(axis=0)

plt.figure(figsize=(11, 3.8))
plt.subplot(1, 2, 1)
for i in range(80):
    plt.plot(t, paths[i], lw=0.5, alpha=0.4)
plt.plot(t, np.sqrt(t), color="red", lw=2)
plt.plot(t, -np.sqrt(t), color="red", lw=2)
plt.xlabel("t")
plt.ylabel("position")
plt.title("Random walk paths")

plt.subplot(1, 2, 2)
plt.plot(t, spread, label="measured std")
plt.plot(t, np.sqrt(t), "--", color="red", lw=2, label="sqrt(t)")
plt.xlabel("t")
plt.ylabel("std of position")
plt.title("Spread grows as sqrt(t)")
plt.legend()
plt.tight_layout()
plt.show()

print(f"t=500  측정 {spread[-1]:.3f}  이론 {np.sqrt(500):.3f}")

## Part 2. 두 가지 방법으로 stationary distribution 구하기

날씨 전이행렬을 쓴다 (맑음 / 흐림 / 비).

### 실습 2 — 방법 A: 행렬을 계속 곱하기

In [ ]:
P = np.array([
    [0.70, 0.20, 0.10],
    [0.30, 0.40, 0.30],
    [0.20, 0.45, 0.35],
])
labels = ["sunny", "cloudy", "rainy"]

v = np.array([1.0, 0.0, 0.0])
for step in range(1, 31):
    # TODO 2: 한 걸음 나아가세요.  힌트: v = v @ P
    pass
    if step in [1, 2, 5, 10, 30]:
        print(f"{step:>3}일 후: {np.round(v, 4)}")

pi_matrix = v
print("\n행렬 곱으로 얻은 stationary:", np.round(pi_matrix, 4))

### 실습 3 — 방법 B: 한 궤적을 아주 오래 돌리기

이번에는 **실제로 날씨를 하루하루 시뮬레이션**해서 각 상태에 며칠 있었는지 센다.

In [ ]:
def simulate_chain(P, n_days, start=0, rng=rng):
    state = start
    visits = np.zeros(len(P), dtype=int)
    for _ in range(n_days):
        visits[state] += 1
        # TODO 3: 현재 상태의 행 P[state] 를 확률로 다음 상태를 뽑으세요
        #         힌트: state = rng.choice(len(P), p=P[state])
        pass
    return visits / n_days


pi_sim = simulate_chain(P, 200_000)

print(f"{'상태':>8}{'시뮬레이션':>14}{'행렬 곱':>12}")
for i, lab in enumerate(labels):
    print(f"{lab:>8}{pi_sim[i]:>14.4f}{pi_matrix[i]:>12.4f}")

🎯 **두 방법이 같은 답을 준다.**

- 방법 A는 "확률을 계산"한 것
- 방법 B는 "실제로 오래 살아본" 것

둘이 일치하는 이유는 **10주차 대수의 법칙**이다.
장기 방문 비율이 stationary distribution으로 수렴한다.

> 이것이 **MCMC의 핵심 아이디어**이기도 하다 —
> 원하는 분포가 stationary가 되도록 chain을 설계하고, 오래 돌려서 표본을 얻는다.

## Part 3. PageRank

웹페이지 5개의 링크 구조에서 **가장 중요한 페이지**를 찾는다.
"무작위 서퍼가 링크를 따라다닐 때 각 페이지에 머무는 시간의 비율"이 곧 순위다.

### 실습 4

In [ ]:
links = np.array([
    [0, 1, 1, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 1, 0],
    [0, 0, 0, 0, 1],
    [0, 0, 1, 1, 0],
], dtype=float)

M = links / links.sum(axis=1, keepdims=True)

d = 0.85
n = len(M)
P_rank = d * M + (1 - d) / n

v = np.ones(n) / n
# TODO 4: 전이행렬을 200번 곱해 수렴시키세요
#         힌트: for _ in range(200): v = v @ P_rank

order = np.argsort(-v)
print("PageRank 순위")
for rank, i in enumerate(order, 1):
    print(f"  {rank}위: 페이지 {i}  ({v[i]:.4f})")

> **링크를 많이 받는 페이지, 그리고 중요한 페이지에서 링크를 받는 페이지**가 높은 순위를 얻는다.
> 구글의 출발점이 오늘 배운 stationary distribution이다.

## Part 4. n-gram 문장 생성기 (재미)

Markov 가정으로 문장을 만들어본다. "앞 단어 하나만 보고 다음 단어를 고른다."

### 실습 5

In [ ]:
text = ("""확률은 불확실성을 다루는 언어다 확률은 데이터를 이해하는 도구다
데이터는 세상을 이해하는 창이다 시뮬레이션은 직관을 만드는 방법이다
직관은 수식을 이해하는 열쇠다 수식은 현상을 설명하는 언어다""").split()

nxt = {}
for a, b in zip(text[:-1], text[1:]):
    # TODO 5: a 다음에 b가 온다는 것을 기록하세요
    #         힌트: nxt.setdefault(a, []).append(b)
    pass

word = "확률은"
out = [word]
for _ in range(12):
    if word not in nxt:
        break
    word = rng.choice(nxt[word])
    out.append(word)

print(" ".join(out))

> 데이터가 6문장뿐이라 어색하지만, **원리는 n-gram 언어모델과 같다.**
> 코퍼스를 키우고 앞 단어를 2~3개 보게 만들면 훨씬 그럴듯해진다.
>
> LLM은 여기서 두 가지를 바꿨다 —
> **앞의 모든 토큰을 보고**(Markov 가정을 깨고), **신경망으로 확률을 계산**한다.

---

## 마무리

- [ ] random walk의 퍼짐이 $\sqrt{t}$ 임을 확인했다
- [ ] 행렬 곱과 장기 시뮬레이션이 같은 답을 준다는 것을 확인했다
- [ ] PageRank가 stationary distribution임을 이해했다

**한 학기 동안 만든 노트북 15개를 잘 보관해두자.**
2학년 과목에서 다시 열어보게 된다.

🎓 **수고했습니다.**